## Глава 8 - Применение машинного обучения для смыслового анализа текста

В нашу эпоху мнения, отзывы и рекомендации людей, высказанные ими в Интернете и социальных сетях, превратились в ценный ресурс для политтехнологов и предпринимателей. Благодаря современным технологиям мы теперь можем максимально эффективно собирать и анализировать такие данные. В этой главе мы займемся изучением одного из применений обработки естественного языка (Natural Language Processing,
NLP) - так называемого анализа эмоциональной окраски текстов, и покажем, как использовать алгоритмы машинного обучения для классификации документов на основе эмоциональной тональности высказываний их авторов. В частности, мы проанализируем набор данных из 50 тыс. отзывов о фильмах из базы данных фильмов в Интернете (lMDb) и построим предиктор, который сможет различать положительные и отрицательные отзывы. 

В этой главе будут рассмотрены следующие темы:\
❖ очистка и подготовка текстовых данных;\
❖ построение векторов признаков из текстовых документов;\
❖ обучение модели, различающей положительные и отрицательные отзывы о фильмах;\
❖ работа с большими наборами текстовых данных с использованием дополнительного
обучения\
❖ извлечение тем из коллекций документов для последующей категоризации. 

### 8.1. Подготовка набора данных с обзорами фильмов на IMDb 

Как уже упоминалось, анализ эмоциональной окраски, иногда также называемый анализом мнений, или анализом настроений, является популярным прикладным применением более масштабных технологий NLP, - он связан со смысловым анализом документов. Популярной задачей в анализе настроений является классификация документов на основе мнений или эмоций авторов в отношении той или иной темы. В этой главе мы будем работать с большим набором обзоров фильмов из базы IMDb, собранным Эндрю Маасом и его коллегами. Набор данных состоит из 50 тыс. обзоров фильмов с четко выраженным мнением авторов, помеченных как положительные или отрицательные, - здесь положительный обзор означает, что фильм получил более шести звезд на IMDb, а отрицательный - что фильм получил менее пяти звезд на IMDb. В следующих разделах мы загрузим этот набор данных, переведем его в формат, пригодный для инструментов машинного обучения, и извлечем значимую информацию из подмножества обзоров фильмов, чтобы построить модель машинного обучения, способную предсказать, понравится или не понравится фильм определенному обозревателю. 

#### 8.1.1. Получение набора данных с обзорами фильмов 

In [16]:
import os
import sys
import tarfile # Позволяет работать с tar-архивами
import time
import urllib.request

source = 'http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz'
target = 'aclImdb_v1.tar.gz'

# Если файл с именем target уже существует в текущей папке, то удалить его.
if os.path.exists(target):
    os.remove(target)

# Определяется функция reporthook. Она будет вызываться автоматически во время скачивания для отчёта о прогрессе.
# Параметры: count — сколько блоков уже скачано; block_size — размер одного блока в байтах; 
# total_size — общий размер файла в байтах
def reporthook(count, block_size, total_size):
    global start_time
    if count == 0:
        start_time = time.time()
        return
    duration = time.time() - start_time # Сколько секунд прошло с начала скачивания до текущего момента.
    progress_size = int(count * block_size) # Сколько байтов уже скачано
    speed = progress_size / (1024.**2 * duration) # Скорость скачивания в мегабайтах в секунду.
    percent = count * block_size * 100. / total_size # Процент выполнения

    # Выводит строку прогресса в консоль.
    sys.stdout.write(f'\r{int(percent)}% | {progress_size / (1024.**2):.2f} MB '
                     f'| {speed:.2f} MB/s | {duration:.2f} sec elapsed')
    # Принудительно отправляет буфер вывода в консоль, чтобы строка появилась немедленно.
    sys.stdout.flush()

#Проверяет два условия: 1.В текущей папке нет папки с именем aclImdb 2.И нет файла aclImdb_v1.tar.gz
if not os.path.isdir('aclImdb') and not os.path.isfile('aclImdb_v1.tar.gz'):
    # Скачивает файл из source и сохраняет его как target. Третий аргумент — функция reporthook, 
    # которая будет вызываться для отчёта о прогрессе.
    urllib.request.urlretrieve(source, target, reporthook)

In [17]:
# Открываем tar-архив.
# target — это имя файла ('aclImdb_v1.tar.gz'), который мы скачали ранее.
# 'r:gz' — режим открытия:
# 'r' — read (чтение)
# ':gz' — указывает, что архив сжат с помощью gzip
# with создаёт контекстный менеджер: архив автоматически закроется после выхода из блока with, 
# даже если произойдёт ошибка.
if not os.path.isdir('aclImdb'):
    with tarfile.open(target, 'r:gz') as tar:
        # extractall() без аргументов извлекает все файлы и папки прямо туда, где находится скрипт.
        tar.extractall()

#### 8.1.2. Преобразование набора данных в более удобный формат 

In [18]:
import pyprind
import pandas as pd
import os
import sys
from packaging import version

basepath = 'aclImdb'
labels = {'pos': 1, 'neg': 0}

# Создаёт прогресс-бар на 50 000 шагов (столько всего файлов в датасете). stream=sys.stdout указывает, 
# куда выводить прогресс (в стандартный вывод — консоль). Если прогресс-бар не отображается, можно сменить поток.
pbar = pyprind.ProgBar(50000, stream=sys.stdout)

df = pd.DataFrame()
# Внешний цикл по двум подпапкам: 'test' и 'train' .
for s in ('test', 'train'):
    # Внутренний цикл по двум папкам с отзывами: 'pos' и 'neg'
    for l in ('pos', 'neg'):
        # Формирует путь к папке с файлами.
        path = os.path.join(basepath, s, l)
        # os.listdir(path) — получает список всех файлов в папке.
        # sorted() — сортирует их по алфавиту (для воспроизводимости порядка).
        # Цикл проходит по каждому файлу-отзыву.
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()
                
            if version.parse(pd.__version__) >= version.parse("1.3.2"):
                x = pd.DataFrame([[txt, labels[l]]], columns=['review', 'sentiment'])
                df = pd.concat([df, x], ignore_index=False)

            else:
                df = df.append([[txt, labels[l]]], 
                               ignore_index=True)
            pbar.update()
df.columns = ['review', 'sentiment']

В этом блоке кода мы сначала инициализировали новый объект индикатора выполнения pbar значением 50 тыс. итераций, что соответствует количеству документов, которые мы намерены прочитать. Используя вложенные циклы for, мы прошлись по подкаталогам train и test в главном каталоге aclimdЬ и прочитали отдельные текстовые файлы из подкаталогов pos и neg, которые в конечном итоге добавили к набору данных df (pandas DataFrame) вместе с целочисленной меткой класса (1 = положительный и 0 = отрицательный).\ 
Поскольку метки классов в собранном наборе данных упорядочены, необходимо перемешать DataFrame с помощью функции permutation из подмодуля np.random- это будет полезно для разделения исходного набора данных на обучающую и тестовую части в последующих разделах, когда мы будем брать данные со своего локального диска напрямую. Для нашего же удобства сохраним собранный и перетасованный набор данных обзоров
в виде СSV-файла: 

In [19]:
import numpy as np

if version.parse(pd.__version__) >= version.parse("1.3.2"):
    df = df.sample(frac=1, random_state=0).reset_index(drop=True)
else:
    np.random.seed(0)
    df = df.reindex(np.random.permutation(df.index))

In [20]:
df.to_csv('movie_data.csv', index=False, encoding='utf-8')

In [21]:
import pandas as pd
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df = df.rename(columns={"0": "review", "1": "sentiment"})
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


In [22]:
df.shape

(50000, 2)

### 8.2. Знакомство с моделью мешка слов

Возможно, вы помните из главы 4, что прежде, чем передать алгоритму машинного обучения категориальные данные, такие как текст или слова, их следует преобразовать в числовую форму. В этом разделе вы познакомитесь с моделью мешка слов (bag-of-words), которая позволяет представлять текст в виде векторов числовых признаков.\
Идея мешка слов довольно проста и вкратце выглядит так:
1. Создаем словарь уникальных токенов - например, слов - из всего набора документов.
2. Для каждого документа строим вектор признаков, который содержит подсчеты того, как часто каждое слово встречается в конкретном документе.

Поскольку уникальные слова в каждом документе представляют собой лишь небольшое подмножество всех слов в словаре, векторы признаков будут в основном состоять из нулей. Такие векторы называются разреженными

#### 8.2.1. Преобразование слов в векторы признаков 

Чтобы построить модель мешка слов на основе количества слов в соответствующих документах, воспользуемся классом CountVectorizer, реализованным в scikit-leam. Как вы увидите в следующем фрагменте кода, countVectorizer берет массив текстовых данных, которые могут быть документами или предложениями, и строит для нас модель мешка слов: 

In [23]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer()
docs = np.array([
        'The sun is shining',
        'The weather is sweet',
        'The sun is shining, the weather is sweet, and one and one is two'])
bag = count.fit_transform(docs)

Вызвав метод fit_transform класса CountVectorizer, мы построили словарь модели мешка слов и преобразовали в разреженные векторы признаков следующие три предложения:\
♦ 'The sun is shining' (Солнце светит);\
♦ 'The weather is sweet' (Погода прекрасная);\
♦ 'The sun is shining, the weather is sweet, and one and one is two' (Солнце светит, погода прекрасная, а один плюс один будет два).\
Выведем на печать содержание словаря, чтобы проиллюстрировать основные понятия мешка слов: 

In [24]:
print(count.vocabulary_)

{'the': 6, 'sun': 4, 'is': 1, 'shining': 3, 'weather': 8, 'sweet': 5, 'and': 0, 'one': 2, 'two': 7}


Как видно из результата выполнения предьщущей команды, лексический словарь (вокабуляр) хранится в словаре Python, который сопоставляет уникальные слова с целочисленными индексами. Теперь выведем только что созданные векторы признаков: 

In [25]:
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]


Каждая позиция индекса в векторах признаков, показанных здесь, соответствует целочисленным значениям, которые хранятся как элементы словаря в словаре countvectorizer. Например, первый признак в позиции индекса 0 отражает частотность слова 'and', которое встречается только в последнем предложении, тогда как слово 'is' в позиции индекса 1 (второй признак в векторах признаков) встречается во всех трех предложениях. Эти значения в векторах признаков также называются необработанными частотами терминов (raw term frequencies): $tf(t, d)$- количество раз, когда термин $t$ встречается в документе $d$. Следует отметить, что в модели мешка слов порядок слов или терминов в предложении или документе не имеет значения. Порядок, в котором частотности терминов появляются в векторе признаков, определяется индексами словаря, которые обычно присваиваются в алфавитном порядке. 

#### 8.2.2. Оценка релевантности слов с помощью частоты термина и обратной частоты документа 

Анализируя текстовые данные, мы часто сталкиваемся со словами, которые встречаются в нескольких документах обоих классов. Эти часто встречающиеся слова обычно не содержат полезной информации, способствующей различению документов. Существует полезный метод под названием обратная частота документа (Term FrequencyInverse Document Frequency, TF-IDF), который можно использовать для понижения веса таких часто встречающихся слов в векторах признаков. Показатель TF-IDF можно определить как произведение частоты термина и обратной частоты документа: 

$$\text{tf-idf}(t,d)=\text{tf (t,d)}\times \text{idf}(t,d)$$

Здесь $tf(t, d)$ - частота термина, о которой мы говорили в предыдущем разделе, а $idf(t, d)$ - обратная частота документа, которую можно рассчитать следующим образом: 

$$\text{idf}(t,d) = \text{log}\frac{n_d}{1+\text{df}(d, t)},$$

Здесь $n_d$ d- общее количество документов, a $df(d, t)$ )- количество документов $d$, содержащих термин $t$. Добавление константы 1 к знаменателю не обязательно и служит лишь для присвоения ненулевого значения терминам, которые не встречаются ни в одном из обучающих примеров. Логарифм применен с тем, чтобы низкие частоты документов не получили слишком большой вес.\
Библиотека scikit-leam реализует еще один преобразователь - класс TfictfTransfonner, который берет необработанные частоты терминов из класса countvectorizer в качестве входных данных и преобразует их в TF-IDF: 

In [26]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer(use_idf=True, norm='l2', smooth_idf=True)
np.set_printoptions(precision=2)
print(tfidf.fit_transform(count.fit_transform(docs))
      .toarray())

[[0.   0.43 0.   0.56 0.56 0.   0.43 0.   0.  ]
 [0.   0.43 0.   0.   0.   0.56 0.43 0.   0.56]
 [0.5  0.45 0.5  0.19 0.19 0.19 0.3  0.25 0.19]]


Как вы видели в предыдущем разделе, слово 'is' имеет наивысшую частотность в третьем предложении, поскольку там наиболее часто встречается. Однако после преобразования того же вектора признаков в TF-IDF слово 'is' теперь связано в третьем предложении с относительно небольшим весом (0.45), поскольку оно также присутствует и в первом, и во втором предложении и, следовательно, вряд ли может содержать какую-либо информацию, отражающую различия предложений.\
Тем не менее если мы вручную вычислим показатель TF-IDF отдельных терминов в наших векторах признаков, то заметим, что TfidfTransformer вычисляет его немного иначе, чем стандартные уравнения, которые мы привели в этой книге ранее. Уравнение для обратной частоты документов, реализованное в scikit-learn, выглядит следующим образом: $\text{idf} (t,d) = log\frac{1 + n_d}{1 + \text{df}(d, t)}$
Точно так же показатель TF-IDF, вычисленный в scikit-leam, немного отличается от уравнения по умолчанию, которое мы определили ранее, и выглядит так: 
$\text{tf-idf}(t,d) = \text{tf}(t,d) \times (\text{idf}(t,d)+1)$
Наличие слагаемого «+ 1 » в этом уравнении связано со значением параметра smooth_idf=True в предыдущем примере кода и позволяет назначать нулевой вес (т.е. idf(t, d) = log(l) = 0) терминам, которые встречаются во всех документах. Хотя обычно принято нормализовать частоты необработанных терминов перед вычислением TF-IDF, класс TfidfTransformer нормализует TF-IDF напрямую. По умолчанию (параметр norm=' l2 ') TfidfTransformer применяет нормализацию L2, которая возвращает вектор длины l путем деления ненормализованного вектора признаков v на его норму L2:
$v_{\text{norm}} = \frac{v}{||v||_2} = \frac{v}{\sqrt{v_{1}^{2} + v_{2}^{2} + \dots + v_{n}^{2}}} = \frac{v}{\big (\sum_{i=1}^{n} v_{i}^{2}\big)^\frac{1}{2}}$
Давайте закрепим понимание принципов работы класса TfidfTransformer на примере и вычислим TF-IDF слова 'is' в третьем предложении. Это слово в третьем предложении имеет частоту термина 3 (tf = 3), и частота этого термина в целом тоже равна 3, поскольку слово 'is' встречается во всех трех предложениях (df = 3). Теперь вычислим обратную частоту документа: 
$\text{idf}("is", d_3) = log \frac{1+3}{1+3} = 0$
Чтобы найти TF-IDF, нам просто нужно добавить 1 к обратной частоте документа и умножить ее на частоту термина: 
$\text{tf-idf}("is", d_3)= 3 \times (0+1) = 3$

Если мы повторим этот расчет для всех терминов в третьем предложении, то получим следующие векторы: [3.39, 3.0, 3.39, 1.29, 1.29, 1.29, 2.0, 1.69, 1.29]. Однако обратите внимание, что значения в этом векторе признаков отличаются от значений, которые нам ранее вернул TfidfTransformer. Последний шаг, который нам осталось сделать, - это L2-нормализация, которую можно применить следующим образом:
$$\text{tfi-df}_{norm} = \frac{[3.39, 3.0, 3.39, 1.29, 1.29, 1.29, 2.0 , 1.69, 1.29]}{\sqrt{[3.39^2, 3.0^2, 3.39^2, 1.29^2, 1.29^2, 1.29^2, 2.0^2 , 1.69^2, 1.29^2]}}$$

$$=[0.5, 0.45, 0.5, 0.19, 0.19, 0.19, 0.3, 0.25, 0.19]$$

$$\Rightarrow \text{tfi-df}_{norm}("is", d3) = 0.45$$
Как видите, теперь результат наших вычислений совпадает с результатом работы класса TfictfTransformer от scikit-learn. Мы успешно закрепили правила расчета TF-IDF на практике и можем смело перейти к следующему разделу и приступить к обработке данных обзоров фильмов. 

#### 8.2.3. Очистка текстовых данных 

Итак, вы уже знаете, что такое модель мешка слов, частота терминов и TF-IDF. Однако прежде, чем построить модель мешка слов, нужно сделать один важный шаг - очистить текстовые данные, удалив из них все нежелательные символы. Чтобы убедиться в необходимости этого шага, отобразим последние 50 символов из первого документа в перетасованном наборе данных обзора фильмов: 

In [27]:
df.loc[0, 'review'][-50:]

'is seven.<br /><br />Title (Brazil): Not Available'

В выведенном фрагменте текста мы видим НТМL-разметку, а также знаки препинания и другие небуквенные символы. Хотя НТМL-разметка почти не содержит полезной семантики, знаки препинания в определенных случаях могут предоставлять полезную дополнительную информацию. Однако сейчас для простоты мы удалим все знаки препинания, кроме символов смайликов, таких как«:)», поскольку они, безусловно, полезны для анализа эмоциональной окраски.
Справиться с этой задачей нам поможет библиотека регулярных выражений (regex) Pythoп с лаконичным названием re: 

In [28]:
import re
def preprocessor(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)',text)
    text = (re.sub('[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    return text

<>:4: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<>:5: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:4: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<>:5: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
C:\Users\Bushi\AppData\Local\Temp\ipykernel_16748\547513637.py:4: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
  emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)',text)
C:\Users\Bushi\AppData\Local\Temp\ipykernel_16748\547513637.py:5: SyntaxWarning: "\W" is an invalid escape sequen

В этом блоке кода с помощью первого регулярного выражения <[^>]*> мы пытаемся удалить из обзоров фильмов всю НТМL-разметку. Хотя многие программисты не
рекомендуют использовать регулярное выражение для анализа HTML, такого регулярного выражения достаточно для очистки этого конкретного набора данных. Поскольку мы заинтересованы лишь в удалении НТМL-разметки и не планируем использовать ее в дальнейшем, простого регулярного выражения будет достаточно. Удалив НТМL-разметку, мы использовали более сложное регулярное выражение для поиска смайликов, которые мы решили временно сохранить. Затем с помощью регулярного выражения [\W]+ мы удалили из текста все символы, не являющиеся словами, и преобразовали текст в символы нижнего регистра. 
В заключение мы добавили временно сохраненные смайлики в конец обрабатываемой строки документа. Кроме того, для единообразия мы удалили из смайликов символ «носа» ,например, дефис в смайлике :-), потому что иногда встречаются смайлики без этого символа. 

In [29]:
preprocessor(df.loc[0, 'review'][-50:])

'is seven title brazil not available'

In [30]:
preprocessor("</a>This :) is :( a test :-)!")

'this is a test :) :( :)'

In [31]:
df['review'] = df['review'].apply(preprocessor)

#### 8.2.4. Получение токенов из документов 

In [33]:
def tokenizer(text):
    return text.split()

tokenizer('runners like running and thus they run')

['runners', 'like', 'running', 'and', 'thus', 'they', 'run']

В контексте токенизации другой полезный метод - определение основы слова (стемминг), т. е. процесс преобразования слова в его корневую морфему. Стемминг позволяет нам сопоставлять родственные слова с одной и той же основой. Первоначальный алгоритм стемминга был разработан в 1979 году Мартином Портером и поэтому известен как алгоритм Портера

In [35]:
from nltk.stem.porter import PorterStemmer

porter = PorterStemmer()
def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

tokenizer_porter('runners like running and thus they run')

['runner', 'like', 'run', 'and', 'thu', 'they', 'run']

Прежде чем перейти непосредственно к обучению модели с использованием метода мешка слов, кратко коснемся другой полезной темы, называемой удалением стоп-слов. Стоп-слова - это всего лишь слова, которые чрезвычайно распространены во всех видах текстов и, вероятно, не несут (или содержат лишь немного) полезной информации, пригодной для различения различных классов документов. Примеры стоп-слов в английском языке: is, has, and, /ike. Удаление стоп-слов особенно полезно, если мы работаем с необработанными или нормализованными частотами терминов, а не с показателями TF-IDF, которые существенно снижают вес часто встречающихся слов. 

In [37]:
import nltk
nltk.download('stopwords') # наборо из 127 английских стоп-слов

from nltk.corpus import stopwords

stop = stopwords.words('english')
[w for w in tokenizer_porter('a runner likes running and runs a lot')
 if w not in stop]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Bushi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['runner', 'like', 'run', 'run', 'lot']

### 8.3. Обучение модели логистической регрессии для классификации документов 

Здесь мы обучим модель логистической регрессии, чтобы классифицировать обзоры фильмов на две категории: положительные и отрицательные - на основе модели мешка слов. Сначала разделим DataFrame очищенных текстовых документов на 25 тыс. документов для обучения и 25 тыс документов для тестирования: 

In [38]:
X_train = df.loc[:25000, 'review'].values
y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
y_test = df.loc[25000:, 'sentiment'].values

Далее воспользуемся объектом GridSearchcv, чтобы найти оптимальный набор параметров для нашей модели лоrистической регрессии с использованием 5-кратной стратифицированной перекрестной проверки: 

Обратите внимание, что для классификатора лоrистической регрессии мы используем решатель liblinear, поскольку с относительно большими наборами данных он может работать лучше, чем выбор по умолчанию (lbfgs). 

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV

tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None)

"""
param_grid = [{'vect__ngram_range': [(1, 1)],
               'vect__stop_words': [stop, None],
               'vect__tokenizer': [tokenizer, tokenizer_porter],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]},
              {'vect__ngram_range': [(1, 1)],
               'vect__stop_words': [stop, None],
               'vect__tokenizer': [tokenizer, tokenizer_porter],
               'vect__use_idf':[False],
               'vect__norm':[None],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]},
              ]
"""

small_param_grid = [{'vect__ngram_range': [(1, 1)],
                     'vect__stop_words': [None],
                     'vect__tokenizer': [tokenizer, tokenizer_porter],
                     'clf__penalty': ['l2'],
                     'clf__C': [1.0, 10.0]},
                    {'vect__ngram_range': [(1, 1)],
                     'vect__stop_words': [stop, None],
                     'vect__tokenizer': [tokenizer],
                     'vect__use_idf':[False],
                     'vect__norm':[None],
                     'clf__penalty': ['l2'],
                  'clf__C': [1.0, 10.0]},
              ]

lr_tfidf = Pipeline([('vect', tfidf),
                     ('clf', LogisticRegression(solver='liblinear'))])

gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid,
                           scoring='accuracy',
                           cv=5,
                           verbose=1,
                           n_jobs=-1)

При инициализации объекта GridSearchcv и его сетки параметров с помощью приведенного кода нам пришлось ограничить количество комбинаций параметров, поскольку большое количество векторов признаков, а также большой словарь, могут сделать поиск по сетке довольно затратным в вычислительном отношении. На стандартном настольном компьютере поиск по сетке может занять 5-10 минут.

В приведенном примере кода мы заменили CountVectorizer и TfidfTransformer на TfidfVectorizer, который фактически объединяет в себе эти два класса. Наш param_grid состоит из двух словарей параметров. В первом словаре мы использовали для вычисления TF-IDS объект TfidfVectorizer с настройками по умолчанию (use_idf=True, smooth _ idf=True и norm='l2'), а во втором словаре установили для этих параметров значения use_idf=False, smooth_idf=False и nornFNone, чтобы обучить модель на основе необработанных частот терминов. Кроме того, для классификатора логистической регрессии мы обучили модели, задав регуляризацию L2 с помощью параметра clf_penalty=l2, и сравнили эффекты от различной степени регуляризации, определив диапазон значений для параметра обратной регуляризации С. В качестве дополнительного упражнения попробуйте добавить в параметры поиска по сетке регуляризацию L1, изменив
'clf_penalty': ['l2'] на 'clf_penalty': ['l2', 'l1'].

In [40]:
gs_lr_tfidf.fit(X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\Bushi\OneDrive\Desktop\Scikit_Learn\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'clf__C': [1.0, 10.0], 'clf__penalty': ['l2'], 'vect__ngram_range': [(1, ...)], 'vect__stop_words': [None], ...}, {'clf__C': [1.0, 10.0], 'clf__penalty': ['l2'], 'vect__ngram_range': [(1, ...)], 'vect__norm': [None], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intCo

После завершения поиска по сетке выведем лучший набор параметров: print(f'Лyчший набор параметров: {gs_lr_tfidf.best_params_}')
Как видите, мы получили наилучшие результаты поиска по сетке, используя обычный tokenizer без стемминга Портера, без библиотеки стоп-слов и TF-IDF в сочетании с классификатором логистической регрессии, который использует регуляризацию L2 со степенью регуляризации C=10.

In [42]:
print(f'Лучшийй набор параметров: {gs_lr_tfidf.best_params_}')

Лучшийй набор параметров: {'clf__C': 10.0, 'clf__penalty': 'l2', 'vect__ngram_range': (1, 1), 'vect__stop_words': None, 'vect__tokenizer': <function tokenizer at 0x00000170067A0EB0>}


Применив лучшую модель в соответствии с результатом поиска по сетке, выведем средние значения оценки точности путем 5-кратной перекрестной проверки на наборе обучающих данных и точности классификации на наборе тестовых данных: 

In [45]:
print(f'Toчнocть CV: {gs_lr_tfidf.best_score_:.3f}')
clf = gs_lr_tfidf.best_estimator_
print(f'Toчнocть на тестовом наборе: {clf.score(X_test, y_test):.3f}')

Toчнocть CV: 0.897
Toчнocть на тестовом наборе: 0.899


<hr>
<hr>

####  Start comment:
    
Please note that `gs_lr_tfidf.best_score_` is the average k-fold cross-validation score. I.e., if we have a `GridSearchCV` object with 5-fold cross-validation (like the one above), the `best_score_` attribute returns the average score over the 5-folds of the best model. To illustrate this with an example:

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

np.random.seed(0)
np.set_printoptions(precision=6)
y = [np.random.randint(3) for i in range(25)]
X = (y + np.random.randn(25)).reshape(-1, 1)

cv5_idx = list(StratifiedKFold(n_splits=5, shuffle=False).split(X, y))
    
lr = LogisticRegression()
cross_val_score(lr, X, y, cv=cv5_idx)

array([0.6, 0.4, 0.6, 0.2, 0.6])

By executing the code above, we created a simple data set of random integers that shall represent our class labels. Next, we fed the indices of 5 cross-validation folds (`cv3_idx`) to the `cross_val_score` scorer, which returned 5 accuracy scores -- these are the 5 accuracy values for the 5 test folds.  

Next, let us use the `GridSearchCV` object and feed it the same 5 cross-validation sets (via the pre-generated `cv3_idx` indices):

In [ ]:
from sklearn.model_selection import GridSearchCV

lr = LogisticRegression()
gs = GridSearchCV(lr, {}, cv=cv5_idx, verbose=3).fit(X, y) 

Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV 1/5] END ..................................., score=0.600 total time=   0.0s
[CV 2/5] END ..................................., score=0.400 total time=   0.0s
[CV 3/5] END ..................................., score=0.600 total time=   0.0s
[CV 4/5] END ..................................., score=0.200 total time=   0.0s
[CV 5/5] END ..................................., score=0.600 total time=   0.0s


As we can see, the scores for the 5 folds are exactly the same as the ones from `cross_val_score` earlier.

Now, the best_score_ attribute of the `GridSearchCV` object, which becomes available after `fit`ting, returns the average accuracy score of the best model:

In [ ]:
gs.best_score_

0.48

As we can see, the result above is consistent with the average score computed with `cross_val_score`.

In [ ]:
lr = LogisticRegression()
cross_val_score(lr, X, y, cv=cv5_idx).mean()

0.48

#### End comment.

<hr>
<hr>

### 8.4. Работа с большими данными: онлайн-алгоритмы и внешнее обучение 

In [ ]:
# Эта ячейка не содержится в книге, но добавлена для удобства, чтобы блокнот можно было запускать, начиная с этого места, без выполнения
# предыдущего кода в этом блокноте

import os
import gzip


if not os.path.isfile('movie_data.csv'):
    if not os.path.isfile('movie_data.csv.gz'):
        print('Please place a copy of the movie_data.csv.gz'
              'in this directory. You can obtain it by'
              'a) executing the code in the beginning of this'
              'notebook or b) by downloading it from GitHub:'
              'https://github.com/rasbt/machine-learning-book/'
              'blob/main/ch08/movie_data.csv.gz')
    else:
        with gzip.open('movie_data.csv.gz', 'rb') as in_f, \
                open('movie_data.csv', 'wb') as out_f:
            out_f.write(in_f.read())

Возможно, вы помните, что еще в главе 2 была введена концепция стохастического градиентного спуска - алгоритма оптимизации, обновляющего веса модели, используя один пример за раз. В этом разделе мы применим функцию partial_fit SGDClassifier библиотеки scikit-leam для потоковой передачи документов непосредственно с нашего локального диска и обучения модели логистической регрессии с использованием небольших мини-пакетов документов. Сначала определим функцию tokenizer, которая очищает необработанные текстовые данные из файла movie_data.csv, созданного в начале этой главы, и разбивает их на однословные токены, попутно удаляя стоп-слова: 

In [46]:
import numpy as np
import re
from nltk.corpus import stopwords

stop = stopwords.words('english')

def tokenizer(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = re.sub('[\W]+', ' ', text.lower()) +\
        ' '.join(emoticons).replace('-', '')
    tokenized = [w for w in text.split() if w not in stop]
    return tokenized

<>:9: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<>:10: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:9: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
<>:10: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
C:\Users\Bushi\AppData\Local\Temp\ipykernel_16748\2294007951.py:9: SyntaxWarning: "\)" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\)"? A raw string is also an option.
  emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
C:\Users\Bushi\AppData\Local\Temp\ipykernel_16748\2294007951.py:10: SyntaxWarning: "\W" is an invalid escape 

Затем определим функцию генератора stream_docs, которая считывает и возвращает один документ за раз: 

In [49]:
def stream_docs(path):
    with open(path, 'r', encoding='utf-8') as csv:
        next(csv)  # skip header
        for line in csv:
            text, label = line[:-3], int(line[-2])
            yield text, label

# проверка
next(stream_docs(path='movie_data.csv'))            

('"In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70\'s, they discover the criminal and a net of power and money to cover the murder.<br /><br />""Murder in Greenwich"" is a good TV movie, with the true story of a murder of a fifteen years old girl that was committed by a wealthy teenager whose mother was a Kennedy. The powerful and rich f

Теперь определим функцию get_minibatch, которая будет получать поток документов из функции stream docs и возвращать нужное количество документов, заданное параметром size: 

In [52]:
def get_minibatch(doc_stream, size):
    docs, y = [], []
    try:
        for _ in range(size):
            text, label = next(doc_stream)
            docs.append(text)
            y.append(label)
    except StopIteration:
        return None, None
    return docs, y

К сожалению, мы не можем использовать объект countVectorizer для внешнего обучения, поскольку он должен хранить в памяти весь словарный запас. Кроме того, TfidfVectorizer хранит в памяти все векторы признаков обучающего набора данных для вычисления обратных частот документа. Однако в scikit-leam реализован еще один удобный векторизатор под названием нashingVectorizer. Он не хранит все данные в памяти и использует прием хеширования с помощью 32-битной функции мurmurHashЗ от Остина Эпплби : 

In [57]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier


vect = HashingVectorizer(decode_error='ignore', 
                         n_features=2**21,
                         preprocessor=None, 
                         tokenizer=tokenizer)

clf = SGDClassifier(loss='log_loss', random_state=1)
doc_stream = stream_docs(path='movie_data.csv')

In [55]:
# from distutils.version import LooseVersion as Version
# from sklearn import __version__ as sklearn_version


Выполнив приведенный код, мы инициализировали HashingVectorizer с помощью функции tokenizer и установили количество признаков равным 2**21. Кроме того, мы повторно инициализировали классификатор логистической регрессии, установив для параметра потерь SGDClassifier значение 'log'. Заметьте, что, выбирая большое количество признаков в нashingVectorizer, мы уменьшаем вероятность возникновения коллизий хешей, но при этом увеличиваем количество коэффициентов в нашей модели логистической регрессии. Наконец-то начинается действительно интересная часть - настроив все вспомогательные функции, мы можем начать обучение на внешних данных с помощью следующего кода: 

In [58]:
import pyprind
pbar = pyprind.ProgBar(45)

classes = np.array([0, 1])
for _ in range(45):
    X_train, y_train = get_minibatch(doc_stream, size=1000)
    if not X_train:
        break
    X_train = vect.transform(X_train)
    clf.partial_fit(X_train, y_train, classes=classes)
    pbar.update()

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:12


Для оценки прогресса обучения мы снова использовали пакет PyPrind. Мы инициализировали объект индикатора выполнения, содержащий 45 делений, а в следующем цикле for перебрали более 45 мини-пакетов документов, где каждый мини-пакет состоит из 1000 документов. Завершив процесс поэтапного обучения, мы задействуем последние 5000 документов для оценки точности нашей модели: 

In [59]:
X_test, y_test = get_minibatch(doc_stream, size=5000)
X_test = vect.transform(X_test)
print(f'Accuracy: {clf.score(X_test, y_test):.3f}')

Accuracy: 0.868


Как видите, точность модели составляет примерно 87%, что немного ниже точности, которой мы достигли в предыдущем разделе, используя поиск по сетке для настройки гиперпараметров. Однако обучение на внешних данных очень эффективно использует память, и его выполнение заняло менее минуты. Наконец, задействуем оставшиеся 5000 документов для обновления модели: 

In [60]:
clf = clf.partial_fit(X_test, y_test)

### 8.5. Моделирование тем с использованием скрытого распределения Дирихле 

Моделирование тем (topic modeling) охватывает широкий перечень задач назначения тем немаркированным текстовым документам. Например, типичным применением является категоризация документов в большом текстовом корпусе газетных статей. В приложениях моделирования тем мы стараемся присвоить газетным статьям метки категорий - например: спорт, финансы, мировые новости, политика и местные новости. Таким образом, с точки зрения широких категорий машинного обучения, которые мы обсуждали в главе 1, моделирование тем можно рассматривать как задачу кластеризации, т. е. разновидность обучения без учителя. В этом разделе мы обсудим популярную технику моделирования тем, называемую скрытым распределением Дирихле (Latent Dirichlet Allocation, LDA). Однако обратите внимание, что хотя для скрытого распределения Дирихле в источниках часто используют аббревиатуру LDA, его не следует путать с обозначаемым той же аббревиатурой линейным дискриминантным анализом (Linear Discriminant Analysis, LDA)- методом контролируемого уменьшения размерности, который был представлен в главе 5. 

#### 8.5.1. Разбор текстовых документов с помощью LDA 

LDA - это генеративная вероятностная модель, которая пытается найти группы слов, часто встречающихся вместе в разных документах. Эти часто встречающиеся слова отражают темы документов, если предположить, что каждый документ состоит из смеси разных слов. Входные данные для LDA предоставляет модель мешка слов, которую мы обсуждали ранее в этой главе.\
Получив в качестве входных данных матрицу мешка слов, LDA разбивает ее на две новые матрицы:\
♦ матрица документ-тема;\
♦ матрица слово-тема.\
LDA разлагает матрицу мешка слов таким образом, чтобы перемножением двух матриц мы могли воспроизвести исходную матрицу мешка слов с наименьшей возможной ошибкой. На практике нас интересуют темы, которые LDA нашел в матрице мешка слов. Единственным недостатком LDA можно считать необходимость заранее определить количество тем - это гиперпараметр, который нужно указывать вручную. 

#### 8.5.2. Реализация LDA в библиотеке scikit-learn 

Здесь для декомпозиции набора обзора фильмов и его классификации по различным темам мы применим реализованный в scikit-leam класс LatentDirichletAllocation. В следующем примере мы ограничиваем анализ I О различными темами и предлагаем читателям самостоятельно поэкспериментировать с гиперпараметрами алгоритма для дальнейшего изучения тем, которые можно найти в этом наборе данных. Начнем с загрузки набора обзоров фильмов в DataFrame pandas из локального файла movie_data.csv, который мы создали в начале этой главы: 

In [ ]:
import pandas as pd

df = pd.read_csv('movie_data.csv', encoding='utf-8')
# the following is necessary on some computers:
df = df.rename(columns={"0": "review", "1": "sentiment"})
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


Затем применим уже знакомый вам countVectorizer для создания матрицы набора слов, которая поступит на вход LDA.
Для удобства мы воспользуемся встроенной библиотекой стоп-слов английского языка scikit-learn в соответствии с параметром stop _ words=' english': 

In [64]:
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer(stop_words='english',
                        max_df=.1,
                        max_features=5000)
X = count.fit_transform(df['review'].values)

Обратите внимание, что мы установили максимальную частоту встречаемости слов в документе на уровне 10% (max_df=.1)- чтобы исключить слова, которые слишком часто встречаются в документах. Причина удаления часто встречающихся слов проста - это могут быть общие слова, содержащиеся во всех документах, которые, следовательно, с меньшей вероятностью связаны с определенной категорией темы того или иного документа. Кроме того, мы уменьшили количество рассматриваемых наиболее часто встречающихся слов до 5000 (max_features=5000)- чтобы ограничить размерность этого набора данных и улучшить вывод, выполняемый LDA. Поскольку значения гиперпараметров max_df=. l и max _ features=5000 выбраны произвольно, читателям рекомендуется настраивать их при сравнении результатов.\
В следующем примере кода показано, как обучить оценщик LatentDirichletAllocation на матрице набора слов и вывести 10 различных тем из документов:

In [65]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=10,
                                random_state=123,
                                learning_method='batch')
X_topics = lda.fit_transform(X)

У становив параметр learning_ method=' batch', мы позволяем оценщику lda делать оценку на основе всех доступных обучающих данных (матрица мешка слов) за одну итерацию, что медленнее, чем альтернативный метод обучения 'online', но может привести к более точным результатам (значение параметра learning_method='online' соответствует онлайн-обучению или мини-пакетному обучению, которое мы обсуждали в главе 2 и ранее в этой главе). 

После обучения LDA у нас теперь есть доступ к атрибуту corrponent_ экземпляра lda, в котором хранится матрица, содержащая важность слова (здесь 5000) для каждой из 10 тем в порядке возрастания: 

In [66]:
lda.components_.shape

(10, 5000)

Чтобы проанализировать результаты, давайте выведем пять самых важных слов для каждой из 10 тем. Значения важности слов ранжируются в порядке возрастания. Следовательно, чтобы вывести первые пять слов, нам нужно отсортировать массив тем в обратном порядке: 

In [68]:
n_top_words = 5
feature_names = count.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    print(f'Topic {(topic_idx + 1)}:')
    print(' '.join([feature_names[i]
                    for i in topic.argsort()\
                        [:-n_top_words - 1:-1]]))

Topic 1:
horror worst script effects budget
Topic 2:
dvd watched video music guy
Topic 3:
war american series history documentary
Topic 4:
game killer murder thriller crime
Topic 5:
kids comedy episode series school
Topic 6:
family woman mother beautiful feel
Topic 7:
role performance comedy john plays
Topic 8:
action horror john effects dr
Topic 9:
book version original read music
Topic 10:
action wife father police james


Основываясь на пяти самых важных словах для каждой темы, можно догадаться, что LDA определила следующие темы:
1. Просто плохие фильмы (не совсем тематическая категория).
2. Фильмы, как-то связанные с сериалами.
3. Военные фильмы.
4. Боевики.
5. Комедии.
6. Фильмы о семье.   
7. Художественные фильмы.
8. Фильмы ужасов. 
9. Фильмы по книгам.
10. Детективные фильмы.

Чтобы убедиться, что категории осмысленно связаны с реальными обзорами, давайте выведем текст отзывов на три фильма из категории «Фильмы ужасов» (это категория 8 с индексом позиции 7): 

In [73]:
horror = X_topics[:, 7].argsort()[::-1]

for iter_idx, movie_idx in enumerate(horror[:3]):
    print(f'\nФильм ужасов #{(iter_idx + 1)}:')
    print(df['review'][movie_idx][:300], '...')


Фильм ужасов #1:
Screamers is an Italian fantasy film (L'Isola degli Uomini Pesce) bought by Roger Corman and released through his New World Pictures. Of course Corman has to carve his initials on it by having one of his lackeys (Dan T. Miller) direct some additional gore footage before he has it released in the sta ...

Фильм ужасов #2:
OZ is the greatest show ever mad full stop.OZ is the greatest show ever mad full stop.OZ is the greatest show ever mad full stop.OZ is the greatest show ever mad full stop.OZ is the greatest show ever mad full stop.OZ is the greatest show ever mad full stop.OZ is the greatest show ever mad full stop ...

Фильм ужасов #3:
This film marked the end of the "serious" Universal Monsters era (Abbott and Costello meet up with the monsters later in "Abbott and Costello Meet Frankentstein"). It was a somewhat desparate, yet fun attempt to revive the classic monsters of the Wolf Man, Frankenstein's monster, and Dracula one "la ...


Using the preceeding code example, we printed the first 300 characters from the top 3 horror movies and indeed, we can see that the reviews -- even though we don't know which exact movie they belong to -- sound like reviews of horror movies, indeed. (However, one might argue that movie #2 could also belong to topic category 1.)

### 8.6. Заключение 

В этой главе вы научились использовать алгоритмы машинного обучения для смыслового анализа текста и классификации текстовых документов на основе их эмоциональной окраски, что является основной задачей анализа настроений в области NLP. Вы не только узнали, как кодировать документ в виде вектора признаков, используя модель мешка слов, но и научились взвешивать частоту термина по релевантности, применяя
показатель TF-IDF. Работа с текстовыми данными бывает весьма затратной в вычислительном отношении из-за больших векторов признаков, которые создаются во время этого процесса, и в последнем разделе вы узнали, как использовать внешнее или постепенное обучение, чтобы избежать загрузки всего набора данных в память компьютера. 

Наконец, вы познакомились с методом моделирования тем на основе скрытого распределения Дирихле и применили его для распределения обзоров фильмов по различным категориям с помощью обучения без учителя. К этому моменту мы рассмотрели множество концепций машинного обучения, лучших
методов и классифицирующих моделей, обучаемых с учителем. 

В следующей главе мы обратимся к другой подкатегории обучения с учителем - регрессионному анализу, который позволит нам прогнозировать переменные результата на непрерывной шкале, - в отличие от категориальных меток классов в классифицирующих моделях, с которыми мы работали до сих пор. 